In [2]:
# !pip install numpy soundfile librosa scipy tqdm

from pathlib import Path
import re, json, warnings, numpy as np
import soundfile as sf, librosa
from scipy.signal import butter, sosfiltfilt, correlate
from tqdm import tqdm

REF_DIR = Path("/Users/moanason/Downloads/Data_REF")   # rec_sXX_NC#.wav (EEG mixdown)
REC_DIR = Path("/Users/moanason/Downloads/Data_REC")   # sXX/p142_sXX_A/B_NC#.wav

REF_RX   = re.compile(r"^rec_s(?P<sess>\d+)_NC(?P<nc>\d+)\.wav$", re.IGNORECASE)
SRC_GLOB = "p*_s{sess:0>2d}_{AB}_NC{nc}.wav"           # inside REC_DIR/sXX

SR_ASSERT_EQUAL = True             # enforce ref/src same SR (you said 48 kHz)
MAX_LAG_SEC     = 1.0              # ± search window in seconds (on envelope)
HOP_LENGTH      = 1024             # envelope hop
N_FFT           = 2048             # envelope FFT
BANDPASS_HZ     = (100, 8000)      # pre-emphasis for envelope; set None to disable
CENTER_WINDOW_S = 8.0              # window around middle used for offset estimation

OUT_SUFFIX      = "_processed.wav"   # output naming
REPORT_SUFFIX   = "_process_report.json"

GLOBAL_CSV_NAME = "offset_align_summary.csv"
GLOBAL_JSON_NAME= "offset_align_summary.json"

warnings.filterwarnings("ignore")

def read_audio(path):
    x, sr = sf.read(path, always_2d=True)
    return x.astype(np.float32), int(sr)

def write_audio(path, x, sr):
    sf.write(path, np.asarray(x, dtype=np.float32), sr)

def to_mono(x):
    return np.mean(x, axis=1) if x.ndim == 2 else x

def bandpass_sos(sr, lo, hi, order=6):
    return butter(order, [lo, hi], btype="band", fs=sr, output="sos")

def onset_envelope(y, sr, hop, n_fft, band):
    mono = to_mono(y)
    if band is not None:
        lo, hi = band
        sos = bandpass_sos(sr, lo, hi)
        mono = sosfiltfilt(sos, mono)
    return librosa.onset.onset_strength(y=mono, sr=sr, hop_length=hop, n_fft=n_fft)

def xcorr_best_lag(ref_env, src_env, max_lag_frames):
    corr = correlate(ref_env, src_env, mode="full")
    lags = np.arange(-len(src_env)+1, len(ref_env))
    keep = (lags >= -max_lag_frames) & (lags <= max_lag_frames)
    corr = corr[keep]; lags = lags[keep]
    return int(lags[np.argmax(corr)])

def integer_shift(x, samples):
    if samples == 0:
        return x
    pad = np.zeros((abs(samples), x.shape[1]), dtype=x.dtype)
    if samples > 0:
        return np.vstack([pad, x])         # delay source -> pad at start
    cut = -samples
    return x[cut:, :] if cut < len(x) else pad  # advance source -> trim start

def pad_or_trim_to_length(x, target_len):
    n = len(x)
    if n == target_len:
        return x
    if n < target_len:
        pad = np.zeros((target_len - n, x.shape[1]), dtype=x.dtype)
        return np.vstack([x, pad])
    return x[:target_len, :]

def coarse_offset_seconds(ref, src, sr, hop, win_s, max_lag_s):
    ref_env = onset_envelope(ref, sr, hop, N_FFT, BANDPASS_HZ)
    src_env = onset_envelope(src, sr, hop, N_FFT, BANDPASS_HZ)
    frames = len(ref_env)
    center = frames // 2
    half = int(round(win_s * sr / (2 * hop)))
    i0 = max(0, center - half); i1 = min(frames, center + half)
    ref_win = ref_env[i0:i1]
    src_win = src_env[i0:i1] if len(src_env) >= len(ref_win) else src_env[:len(ref_win)]
    max_lag_frames = int(round(max_lag_s * sr / hop))
    lag_frames = xcorr_best_lag(ref_win, src_win, max_lag_frames)
    return lag_frames * hop / sr, dict(lag_frames=int(lag_frames), hop=hop, sr=sr)

import numpy as np
from scipy.signal import correlate

def _onset_env(y, sr, hop=1024, n_fft=2048, band=(100, 8000)):
    mono = y.mean(axis=1) if y.ndim == 2 else y
    if band is not None:
        from scipy.signal import butter, sosfiltfilt
        sos = butter(6, [band[0], band[1]], btype="band", fs=sr, output="sos")
        mono = sosfiltfilt(sos, mono)
    # spectral-flux like onset strength
    import librosa
    return librosa.onset.onset_strength(y=mono, sr=sr, hop_length=hop, n_fft=n_fft)

def _ncc_best_lag(a, b, max_lag_frames):
    # normalised cross-correlation around 0
    a = (a - a.mean()) / (a.std() + 1e-8)
    b = (b - b.mean()) / (b.std() + 1e-8)
    cc = correlate(a, b, mode="full")
    lags = np.arange(-len(b)+1, len(a))
    keep = (lags >= -max_lag_frames) & (lags <= max_lag_frames)
    cc = cc[keep]; lags = lags[keep]
    return int(lags[np.argmax(cc)])

def robust_offset_seconds(ref, src, sr,
                          hop=1024, n_fft=2048,
                          max_lag_s=1.5,
                          win_s=8.0, step_s=2.0,
                          activity_pct=60.0,
                          bands=((150,800),(800,3000),(3000,8000))):
    """
    Multi-window, activity-gated, multi-band median offset (seconds).
    Positive return value => src must be delayed by that many seconds.
    """
    best_band = None
    best_score = -np.inf
    best_lags_sec = None

    for band in bands:
        ref_env = _onset_env(ref, sr, hop, n_fft, band)
        src_env = _onset_env(src, sr, hop, n_fft, band)

        # reference activity mask
        thr = np.percentile(ref_env, activity_pct)
        active = ref_env > thr

        # windowing in envelope frames
        win = int(round(win_s * sr / hop))
        step = int(round(step_s * sr / hop))
        max_lag = int(round(max_lag_s * sr / hop))

        lags = []
        scores = []
        for i0 in range(0, len(ref_env) - win + 1, step):
            i1 = i0 + win
            # keep windows with sufficient activity
            if active[i0:i1].mean() < 0.25:
                continue
            a = ref_env[i0:i1]
            b = src_env[i0:i1] if len(src_env) >= i1 else src_env[i0:min(len(src_env), i1)]
            if len(b) < len(a):
                a = a[:len(b)]
            if len(a) < max(32, win//4):
                continue
            lag = _ncc_best_lag(a, b, max_lag)
            lags.append(lag)
            # record the NCC peak value as a crude quality score
            # recompute short NCC around lag
            cc = np.corrcoef(a, np.roll(b, lag))[0,1]
            scores.append(cc)

        if len(lags) == 0:
            continue

        # choose this band if its median NCC is best
        median_score = float(np.median(scores))
        if median_score > best_score:
            best_score = median_score
            best_band = band
            best_lags_sec = np.array(lags) * hop / sr

    if best_lags_sec is None or len(best_lags_sec) == 0:
        # fallback to single-window coarse method
        return coarse_offset_seconds(ref, src, sr, hop, win_s, max_lag_s)

    # robust central tendency of lags
    lag_sec = float(np.median(best_lags_sec))
    # also return dispersion for QC if you want
    return lag_sec

def align_one_to_ref(ref_path, src_path):
    ref, sr_ref = read_audio(ref_path)
    src, sr_src = read_audio(src_path)
    if SR_ASSERT_EQUAL and sr_ref != sr_src:
        raise ValueError(f"Sample rate mismatch: ref {sr_ref} vs src {sr_src}")
    sr = sr_ref

    # 1) coarse lag from onset envelopes (positive => src must be delayed)
    lag_sec = robust_offset_seconds(ref, src, sr,
                                hop=HOP_LENGTH, n_fft=N_FFT,
                                max_lag_s=MAX_LAG_SEC,
                                win_s=CENTER_WINDOW_S, step_s=2.0,
                                activity_pct=60.0)
    meta = {"method": "robust_multiwindow", "lag_seconds": lag_sec}
    lag_samples = int(round(lag_sec * sr))

    # 2) shift
    shifted = integer_shift(src, lag_samples)

    # 3) pad/trim to reference length
    out = pad_or_trim_to_length(shifted, len(ref))

    # QC: envelope correlation improvement
    env_ref = onset_envelope(ref, sr, HOP_LENGTH, N_FFT, BANDPASS_HZ)
    env_src = onset_envelope(src, sr, HOP_LENGTH, N_FFT, BANDPASS_HZ)
    env_out = onset_envelope(out, sr, HOP_LENGTH, N_FFT, BANDPASS_HZ)
    m = min(len(env_ref), len(env_src), len(env_out))
    corr_before = float(np.corrcoef(env_ref[:m], env_src[:m])[0,1])
    corr_after  = float(np.corrcoef(env_ref[:m], env_out[:m])[0,1])

    report = {
        "ref_path": str(ref_path),
        "src_path": str(src_path),
        "sr": sr,
        "applied_shift_samples": int(lag_samples),
        "applied_shift_seconds": float(lag_samples / sr),
        "coarse_meta": meta,
        "env_corr_before": corr_before,
        "env_corr_after":  corr_after,
        "improvement": float(corr_after - corr_before),
    }
    return out, sr, report

def find_triplets():
    for ref in sorted(REF_DIR.glob("rec_s*_NC*.wav")):
        m = REF_RX.match(ref.name)
        if not m:
            continue
        sess = int(m.group("sess"))
        nc   = int(m.group("nc"))
        sess_dir = REC_DIR / f"s{sess:0>2d}"
        a_list = sorted(sess_dir.glob(SRC_GLOB.format(sess=sess, AB="A", nc=nc)))
        b_list = sorted(sess_dir.glob(SRC_GLOB.format(sess=sess, AB="B", nc=nc)))
        a = a_list[0] if a_list else None
        b = b_list[0] if b_list else None
        yield f"s{sess:0>2d}", f"{nc}", ref, a, b

# batch process
rows, problems = [], []
triplets = list(find_triplets())
print(f"Found {len(triplets)} reference files.")

import csv
for sess, nc, ref_path, a_path, b_path in tqdm(triplets, desc="Aligning by offset"):
    for lab, src_path in (("A", a_path), ("B", b_path)):
        if src_path is None:
            problems.append({"session": sess, "nc": nc, "which": lab, "error": "missing source"})
            continue
        try:
            out, sr, rep = align_one_to_ref(ref_path, src_path)
            out_path = src_path.with_suffix("").as_posix() + OUT_SUFFIX
            write_audio(out_path, out, sr)

            rep_path = src_path.with_suffix("").as_posix() + f"_{lab}" + REPORT_SUFFIX
            with open(rep_path, "w") as f:
                json.dump(rep, f, indent=2)

            rows.append({
                "session": sess, "nc": nc, "which": lab,
                "ref": str(ref_path), "src": str(src_path), "out": out_path,
                "sr": sr,
                "applied_shift_samples": rep["applied_shift_samples"],
                "applied_shift_seconds": rep["applied_shift_seconds"],
                "env_corr_before": rep["env_corr_before"],
                "env_corr_after":  rep["env_corr_after"],
                "improvement": rep["improvement"],
            })
        except Exception as e:
            problems.append({"session": sess, "nc": nc, "which": lab, "error": repr(e)})

csv_path = REC_DIR / GLOBAL_CSV_NAME
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["session","nc","which","ref","src","out","sr","applied_shift_samples","applied_shift_seconds","env_corr_before","env_corr_after","improvement"])
    w.writeheader()
    for r in rows:
        w.writerow(r)

summary = {
    "n_ok": len(rows),
    "n_problem": len(problems),
    "mean_shift_ms": float(np.mean([r["applied_shift_seconds"]*1000 for r in rows])) if rows else None,
    "median_shift_ms": float(np.median([r["applied_shift_seconds"]*1000 for r in rows])) if rows else None,
    "mean_env_corr_improvement": float(np.mean([r["improvement"] for r in rows])) if rows else None,
    "problems_preview": problems[:20],
}
with open(REC_DIR / GLOBAL_JSON_NAME, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print("  CSV ->", csv_path)
print("  JSON->", REC_DIR / GLOBAL_JSON_NAME)

Found 40 reference files.


Aligning by offset: 100%|██████████| 40/40 [05:50<00:00,  8.76s/it]

Saved:
  CSV -> /Users/moanason/Downloads/Data_REC/offset_align_summary.csv
  JSON-> /Users/moanason/Downloads/Data_REC/offset_align_summary.json
